# 24 · Prototype chạy thật: OCR trực tiếp + model — còn đúng số đã báo cáo không, nhanh tới đâu? (W13 bước 8, R-OCR · R-LIVE)

**Vấn đề:** notebook 23 kiểm backend với OCR lấy từ **cache** (R-PAR 100%). Giao diện thật thì OCR **trực tiếp** từng ảnh, ở một tiến trình riêng (`src/prototype/ocr_worker.py`, tránh xung đột Paddle GPU ↔ torch). Nếu OCR trực tiếp đọc khác cache thì prompt khác, và câu trả lời của demo có thể khác số trong khoá luận.

**Notebook này:**
1. Cài thư viện cho **cả** Qwen (transformers 5.17) **và** OCR (PaddleOCR 3.7 + Paddle GPU 3.3.1 + VietOCR) trong cùng phiên, giữ torch nguyên vẹn (quy trình notebook 19).
2. Khởi động OCR tiến trình riêng, kiểm engine **trùng** engine đã dựng cache.
3. Chạy **đúng đường của giao diện** (`prepare_image` → `Backend.ask`, OCR trực tiếp) trên 200 ảnh test mỗi tập (seed 42), so với cache và run eval W11.
4. Ảnh thật của nhóm (nếu đã chuẩn bị trên Drive).
5. Mở giao diện bằng link công khai trong 20 phút để thử trên điện thoại / quay video demo.
6. Tự trả máy.

**Luật ghi trước** (`docs/W13/README.md`, mục "Bước 8"):
| Luật | ĐẠT khi |
|---|---|
| **R-OCR** | văn bản OCR vào prompt của OCR trực tiếp **trùng hệt** cache ở ≥ 98% ảnh, từng tập |
| **R-LIVE** | câu trả lời qua đường giao diện **trùng hệt** run eval W11 ở ≥ 97% câu **và** EM không thấp hơn W11 quá 1,5 điểm, từng tập (bỏ câu có/không) |

Chỉ mô tả: latency p50/p95 (OCR / model / tổng), thời gian khởi động, VRAM, ảnh thật.

**Runtime:** **L4** (Runtime → Change runtime type → L4). Thời gian ≈ 50 phút (≈ 1,3 CU), trong đó 20 phút là giao diện mở để thử.

**Cách chạy:** Runtime → **Run all**. Notebook chạy một mạch tới cuối, kể cả khi luật KHÔNG ĐẠT (kết quả vẫn được ghi lại). Chỉ dừng sớm nếu cài đặt hỏng; khi đó cell báo `❌` kèm lý do.

## 1. Thiết lập môi trường

Mount Drive, lấy code mới nhất từ GitHub, kiểm runtime là L4.

*Kỳ vọng: commit mới nhất (`W13 step 8 …`), `Runtime: NVIDIA L4`, `✅ Thiết lập xong`.*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, hashlib, subprocess, glob
if not os.path.exists('/content/ViVQA-VLM/.git'):
    !cd /content && git clone -q https://github.com/PLHGNAOH/ViVQA-VLM.git
%cd /content/ViVQA-VLM
!git pull -q origin main
!git log -1 --oneline
sys.path.insert(0, '/content/ViVQA-VLM')
from src.eval.prototype_live import LIVE_VERSION, LIVE_RUN, OUT_NAME, REAL_RUN
from src.prototype.app import APP_VERSION
assert LIVE_VERSION == 'prototype-live-v1' and APP_VERSION == 'prototype-app-v1', 'Code cũ -> git pull chưa lấy được commit bước 8'

DRIVE_ROOT = '/content/drive/MyDrive/ViVQA-VLM'
EXP_ROOT = f'{DRIVE_ROOT}/experiments'
OUT = f'{EXP_ROOT}/{OUT_NAME}'
PHOTOS = f'{DRIVE_ROOT}/data/real_photos_w13'          # ảnh thật của nhóm + cau_hoi.csv (tuỳ chọn)
UI_MINUTES = 20                                        # thời gian mở giao diện (phút); 0 = không mở

def run_cmd(cmd, check=True):
    # Chạy lệnh, in log trực tiếp; lỗi -> DỪNG notebook (trừ khi check=False: trả mã lỗi)
    print('$', cmd)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f'Lệnh lỗi (mã {rc}) -> xem log ở trên, gửi Claude')
    return rc

GPU = subprocess.run('nvidia-smi --query-gpu=name --format=csv,noheader', shell=True,
                     capture_output=True, text=True).stdout.strip() or 'CPU'
print('Runtime:', GPU)
if 'L4' not in GPU:
    raise SystemExit('❌ Notebook 24 phải chạy trên L4 (cùng GPU với run eval W11). '
                     'Runtime -> Change runtime type -> L4, rồi Run all.')
print('✅ Thiết lập xong')

## 2. Cài thư viện cho Qwen **và** OCR trong cùng phiên (≈ 6–8 phút)

**Vấn đề (W08, notebook 10/19):** `paddlepaddle-gpu` bản CUDA 12.6 cài kèm NCCL của CUDA 12, **ghi đè** file NCCL mà torch của Colab đang dùng, làm `import torch` hỏng.

**Cell này làm theo đúng quy trình đã chạy được ở notebook 19:**
1. Cài thư viện của Qwen (cùng pin với notebook 23: `transformers==5.17.0`), Gradio (`6.31.0`), VietOCR (`--no-deps`).
2. Chụp danh sách thư viện `nvidia-*` của torch → cài `paddleocr==3.7.0` + `paddlepaddle-gpu==3.3.1` (cùng phiên bản đã dựng cache; thử tối đa 3 lần) → cài lại NCCL của torch.
3. Kiểm ở tiến trình riêng: torch thấy GPU **và** Paddle chạy GPU → `OCR_DEVICE = gpu`. Không được thì gỡ Paddle GPU, khôi phục torch, cài Paddle CPU → `OCR_DEVICE = cpu` (R-OCR sẽ đo xem OCR CPU có còn trùng cache không).
4. Kiểm lần cuối: torch + transformers 5.17 + peft + bitsandbytes còn tốt.

*Kỳ vọng: `torch OK trước khi cài`, `paddleocr 3.7.0 | paddle 3.3.1`, `OCR_DEVICE = gpu` (hoặc `cpu` kèm lý do), `✅ torch + transformers 5.17.0 vẫn tốt`.*

In [ ]:
def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr)[-1500:]

def nvidia_pkgs():
    out = subprocess.run('pip list --format=json', shell=True, capture_output=True, text=True).stdout
    return {p['name'].lower(): p['version'] for p in json.loads(out) if p['name'].lower().startswith('nvidia-')}

def torch_ok():
    return sh('python -c "import torch; assert torch.cuda.is_available(); torch.zeros(1).cuda()"')[0] == 0

def paddle_gpu_ok():
    return sh('python -c "import paddle; assert paddle.device.is_compiled_with_cuda(); paddle.utils.run_check()"')[0] == 0

def restore(pkgs):                       # cài lại ĐÚNG phiên bản cũ, không kéo thêm gì
    if pkgs:
        code_, log = sh('pip install -q --force-reinstall --no-deps ' + ' '.join(f'{n}=={v}' for n, v in pkgs.items()))
        assert code_ == 0, log

assert torch_ok(), '❌ torch đã hỏng từ trước -> Runtime -> Disconnect and delete runtime, rồi Run all lại'
print('torch OK trước khi cài')
code_, log = sh('pip install -q transformers==5.17.0 accelerate qwen-vl-utils bitsandbytes peft huggingface_hub pyyaml gradio==6.31.0')
assert code_ == 0, log
code_, log = sh('pip install -q --no-deps vietocr==0.3.13 && pip install -q gdown einops')
assert code_ == 0, log
code_, log = sh('pip install -q paddleocr==3.7.0')
assert code_ == 0, log
BEFORE = nvidia_pkgs()

for attempt in range(1, 4):
    code_gpu, log_gpu = sh('pip install -q --timeout 180 --retries 5 paddlepaddle-gpu==3.3.1 '
                           '-i https://www.paddlepaddle.org.cn/packages/stable/cu126/')
    if code_gpu == 0:
        break
    print(f'⚠ Lần {attempt}/3: cài Paddle GPU lỗi -> thử lại. Đuôi log:', log_gpu[-200:].replace(chr(10), ' '))
NEW = sorted(set(nvidia_pkgs()) - set(BEFORE))
restore({n: v for n, v in BEFORE.items() if 'nccl' in n})
if not torch_ok():
    print('torch vẫn hỏng sau khi khôi phục NCCL -> khôi phục toàn bộ thư viện nvidia-* của torch')
    restore(BEFORE)

if code_gpu == 0 and torch_ok() and paddle_gpu_ok():
    OCR_DEVICE = 'gpu'
else:
    print('⚠ Paddle GPU không chạy chung được với torch' if code_gpu == 0 else '⚠ Cài Paddle GPU lỗi:\n' + log_gpu[-400:])
    print('-> gỡ Paddle GPU, khôi phục thư viện torch, cài Paddle CPU')
    sh('pip uninstall -y -q paddlepaddle-gpu ' + ' '.join(NEW))
    restore(BEFORE)
    code_, log = sh('pip install -q paddlepaddle')
    assert code_ == 0, log
    OCR_DEVICE = 'cpu'

assert torch_ok(), '❌ torch hỏng sau khi cài Paddle -> gửi toàn bộ output cell này cho Claude'
rc, out = sh('python -c "import transformers, peft, bitsandbytes, qwen_vl_utils, gradio; '
             'print(transformers.__version__, gradio.__version__)"')
assert rc == 0 and out.strip().startswith('5.17.0'), f'❌ thư viện Qwen hỏng sau khi cài Paddle:\n{out}'
print(sh('python -c "import paddleocr, paddle; print(\'paddleocr\', paddleocr.__version__, \'| paddle\', paddle.__version__)"')[1])
print('OCR_DEVICE =', OCR_DEVICE, '| ✅ torch + transformers', out.split()[0], 'vẫn tốt | gradio', out.split()[1])

## 3. Kiểm tra trước khi chạy (≈ 2 phút)

- **(a)** Selftest 4 script (offline): `ocr_worker`, `app` (lần này có Gradio → kiểm cả dựng giao diện), `prototype_live`, `prototype_check`.
- **(b)** Ba luật bước 4–6 trong `prototype_check_summary.json` đều ĐẠT (giao diện tự từ chối chạy model thật nếu không).
- **(c)** Run eval W11 của SH có trên Drive; adapter SH trên Drive **trùng** dấu vân tay run W11.
- **(d)** OCR cache hai tập có trên Drive.
- **(e)** **Khởi động OCR thật ở tiến trình riêng**: engine phải trùng engine đã dựng cache (`paddleocr`, `det_model`, `rec_model`, `vietocr_config`, `lang`); đọc thử 1 ảnh test.

*Kỳ vọng: `✅ selftest OK (4 script)`, `luật bước 4–6: ĐẠT ×3`, 2 dòng `mốc W11 … ✅`, `adapter SH = run W11 ✅`, 2 dòng `OCR cache … ✅`, `engine OCR = cache ✅` (có thể kèm dòng ghi chú khác device / phiên bản Paddle), `đọc thử: … dòng chữ`.*

In [ ]:
# (a) selftest
musts = {'src.prototype.ocr_worker': ['selftest ocr_worker OK'],
         'src.prototype.app': ['selftest app OK', 'dựng Gradio'],
         'src.eval.prototype_live': ['selftest prototype_live OK'],
         'src.eval.prototype_check': ['prototype_check selftest OK']}
for mod, need in musts.items():
    o = subprocess.run(f'python -m {mod} --selftest', shell=True, capture_output=True, text=True)
    for m in need:
        assert m in o.stdout, (mod, m, o.stdout[-800:], o.stderr[-800:])
print('✅ selftest OK (4 script)')

# (b) luật bước 4–6
from src.prototype.app import load_ui_settings, require_verified, load_ocr_selection
ST = load_ui_settings()
require_verified(ST)
print('luật bước 4–6:', ST['verdicts'])

# (c) mốc W11 + adapter
from src.eval.prototype_check import REF_RUN
from src.eval.run_baseline import adapter_fingerprint
for ds, run in REF_RUN.items():
    assert os.path.exists(f'{EXP_ROOT}/{run}/predictions.json'), f'thiếu {run}/predictions.json trên Drive -> báo Claude'
    print(f'mốc W11 {ds}: {run} ✅')
ref = json.load(open(f"{EXP_ROOT}/{REF_RUN['vivqa']}/metrics.json", encoding='utf-8'))['meta']['adapter']
AD_SH, AD_SHA = ref['path'], ref['weights_sha256']
assert adapter_fingerprint(AD_SH)['weights_sha256'] == AD_SHA, 'adapter SH trên Drive KHÁC run W11 -> dừng, báo Claude'
print(f'adapter SH = run W11 ✅ ({AD_SHA[:12]}…)')

# (d) OCR cache
from src.data.ocr import default_cache_path, load_ocr_cache, lines_to_texts
SEL = load_ocr_selection()
for ds in ('vivqa', 'vitextvqa_official'):
    p = default_cache_path(DRIVE_ROOT, ds, SEL['backend'])
    assert os.path.exists(p), f'thiếu OCR cache {p}'
    print(f'OCR cache {ds} ✅')

# (e) OCR thật ở tiến trình riêng
from src.prototype.ocr_worker import OCRWorkerClient, check_engine_matches_cache
os.makedirs(OUT, exist_ok=True)
with OCRWorkerClient(backend=SEL['backend'], device=OCR_DEVICE, log_path=f'{OUT}/ocr_smoke.log') as c:
    chk = check_engine_matches_cache(c.info)
    print('engine OCR:', {k: c.info.get(k) for k in ('paddleocr', 'paddlepaddle', 'device', 'det_model', 'rec_model')})
    if chk['device_note']:
        print('ghi chú:', chk['device_note'])
    assert chk['ok'], f"❌ engine OCR khác engine đã dựng cache: {chk['diff']} -> gửi Claude output cell này"
    print('engine OCR = cache ✅')
    cache = load_ocr_cache(default_cache_path(DRIVE_ROOT, 'vivqa', SEL['backend']))
    img = next(k for k, v in cache['images'].items() if v.get('lines'))
    src_img = glob.glob(f'{DRIVE_ROOT}/data/vivqa/images/{img}')
    if src_img:
        live = lines_to_texts({'lines': c.lines(src_img[0])}, SEL['max_chars'], SEL['min_conf'])[0]
        ref_t = lines_to_texts(cache['images'][img], SEL['max_chars'], SEL['min_conf'])[0]
        print(f'đọc thử {img}: {len(live)} dòng chữ | trùng cache: {live == ref_t}')

## 4. Chạy đúng đường của giao diện trên 200 ảnh test mỗi tập (≈ 15–25 phút)

`src.eval.prototype_live --run`:
- Chọn 200 ảnh mỗi tập (sắp tên ảnh, xáo bằng seed 42, lấy 200 ảnh đầu) và **mọi** câu hỏi của các ảnh đó.
- Mỗi câu: `prepare_image` (ảnh dataset đi nguyên file) → `Backend.ask` (OCR trực tiếp ở tiến trình riêng → prompt → Qwen + adapter SH) — đúng đường giao diện dùng.
- Ghi từng câu lên Drive ngay khi xong. **Bị ngắt thì Run all lại**: câu đã có tự bỏ qua.
- Lần đầu chép ảnh ViVQA và giải nén ảnh ViTextVQA về `/content` (≈ 10 phút, như notebook 23).

*Kỳ vọng: `[live] mẫu: …` (≈ 200 ảnh mỗi tập), `[live] khởi động … OCR {"ok": true …}`, các dòng `[live] 50/…`, cuối cùng `== xong … câu`.*

In [ ]:
if os.path.exists(f'{EXP_ROOT}/{LIVE_RUN}/meta.json'):
    print('⏭ đã chạy xong trước đó -> bỏ qua')
else:
    run_cmd(f'python -u -m src.eval.prototype_live --run --drive_root {DRIVE_ROOT} --adapter "{AD_SH}" '
            f'--expect_adapter_sha {AD_SHA} --ocr_device {OCR_DEVICE}')

## 5. Phân tích + phán quyết (R-OCR · R-LIVE) + tốc độ

*Kỳ vọng: 3 mục, 2 dòng `➜ **R-OCR: …**`, `➜ **R-LIVE: …**`, bảng latency. File `prototype_live_summary.json` + `.md` nằm trong `experiments/W13_prototype_live_L4_seed42/`. Luật KHÔNG ĐẠT thì notebook **vẫn chạy tiếp** (kết quả đã ghi lại).*

In [ ]:
run_cmd(f'python -m src.eval.prototype_live --analyze --drive_root {DRIVE_ROOT} --out {OUT}')

## 6. Ảnh thật của nhóm (tuỳ chọn, ≈ 2 phút)

Chạy nếu trên Drive có thư mục `ViVQA-VLM/data/real_photos_w13/` chứa ảnh **và** file `cau_hoi.csv` (mẫu: `docs/W13/realphotos_cau_hoi_mau.csv`; cột `anh`, `cau_hoi`, `dap_an` — đáp án có thể bỏ trống, nhiều cách viết thì tách bằng `|`). Không có thì cell tự bỏ qua.

Ảnh **không** được đưa lên GitHub; kết quả (câu hỏi, câu trả lời, mức, thời gian) ghi vào `experiments/W13_REAL_sh_realphotos_bf16_L4/` trên Drive.

*Kỳ vọng: bảng `# W13 bước 8 — ảnh thật của nhóm`, hoặc `⏭ chưa có ảnh thật -> bỏ qua`.*

In [ ]:
CSV = f'{PHOTOS}/cau_hoi.csv'
if os.path.exists(CSV):
    run_cmd(f'python -u -m src.eval.prototype_live --realphotos --photos_dir "{PHOTOS}" --questions_csv "{CSV}" '
            f'--drive_root {DRIVE_ROOT} --adapter "{AD_SH}" --expect_adapter_sha {AD_SHA} --ocr_device {OCR_DEVICE}',
            check=False)
else:
    print(f'⏭ chưa có ảnh thật ({CSV}) -> bỏ qua')

## 7. Mở giao diện để thử trên điện thoại / quay video demo (20 phút)

Cell in ra một dòng **`Running on public URL: https://….gradio.live`**. Mở link đó trên điện thoại hoặc máy tính, tải ảnh, gõ câu hỏi. Có thể quay màn hình làm **video demo dự phòng** (bước 9).

- Ví dụ sẵn: 4 câu đầu tiên của mẫu ở mục 4 (2 ViTextVQA có chữ, 2 ViVQA) — lấy theo thứ tự, không chọn lọc.
- Link công khai: ai có link đều dùng được trong lúc cell chạy → **không chia sẻ link**. Giao diện không lưu ảnh / câu hỏi; chỉ ghi số đo thời gian vào `ui_log.jsonl`.
- Sau `UI_MINUTES` phút (mặc định 20) giao diện tự tắt và notebook chạy tiếp tới bước trả máy. Muốn tắt sớm: bấm nút ■ (Stop) của cell này rồi chạy tay cell mục 8.

*Kỳ vọng: `Khởi động: {…}` rồi `Running on public URL: https://….gradio.live`.*

In [ ]:
if UI_MINUTES > 0:
    recs = [json.loads(l) for l in open(f'{EXP_ROOT}/{LIVE_RUN}/records.jsonl', encoding='utf-8') if l.strip()]
    from src.eval.run_baseline import prepare_images
    ex = []
    for ds, k in (('vitextvqa_official', 2), ('vivqa', 2)):
        root = prepare_images(ds, DRIVE_ROOT)
        rr = [r for r in recs if r['dataset'] == ds and (r['n_ocr_lines'] > 0 or ds == 'vivqa')][:k]
        ex += [{'image': os.path.join(root, r['image']), 'question': r['question']} for r in rr]
    json.dump(ex, open('/content/ui_examples.json', 'w', encoding='utf-8'), ensure_ascii=False)
    print(f'{len(ex)} ví dụ | mở giao diện {UI_MINUTES} phút')
    !timeout -s INT {UI_MINUTES}m python -u -m src.prototype.app --adapter "{AD_SH}" --expect_adapter_sha {AD_SHA} --ocr_device {OCR_DEVICE} --ocr_log {OUT}/ui_ocr_worker.log --examples_json /content/ui_examples.json --log_jsonl {OUT}/ui_log.jsonl --share
    print('Giao diện đã tắt.')
else:
    print('UI_MINUTES = 0 -> không mở giao diện')

## 8. Tự trả máy L4

Kết quả đã nằm trên Drive. Nếu notebook dừng sớm vì lỗi thì máy **không** tự trả: đọc lỗi xong, tự bấm **Runtime → Disconnect and delete runtime**.

*Kỳ vọng: `✅ Xong toàn bộ -> tự trả máy …`.*

In [ ]:
AUTO_RELEASE = True
if AUTO_RELEASE:
    print('✅ Xong toàn bộ -> tự trả máy để ngừng tính CU (output phía trên vẫn giữ trong tab).')
    from google.colab import runtime
    runtime.unassign()
else:
    print('AUTO_RELEASE = False -> nhớ tự Disconnect and delete runtime khi xong.')

## 9. ⏸ DỪNG

Nhắn Claude **"xong"** + ảnh CU (Claude tự đọc kết quả trên Drive), hoặc gửi output **mục 3** và **mục 5**. Kết quả nằm trong `experiments/W13_LIVE_*`, `W13_prototype_live_L4_seed42/` và (nếu có) `W13_REAL_*` trên Drive.